# ML Fundamentals

The concepts that turn Python skills into machine learning: how models learn from data, the end-to-end workflow, the core algorithms (intuition over math), and — most importantly — how to tell whether a model is actually any good.

**Topics covered in this notebook**

1. NumPy & Pandas (the bridge from Python to ML)
2. What ML actually is
3. The core ML workflow
4. Key algorithms (intuition)
5. Evaluation
6. Overfitting & generalization

**1. NumPy & Pandas (the bridge)**

NumPy provides fast numerical arrays — the vectors and matrices every ML concept is expressed in. Pandas provides labeled tabular data (`DataFrame`), where rows are samples and columns are features. They sit exactly between "Python" and "ML", so don't skip them.

In [1]:
import numpy as np
import pandas as pd

# NumPy: fast numerical arrays - vectors and matrices
vec = np.array([1, 2, 3])
mat = np.array([[1, 2], [3, 4]])
print("vector * 2 :", vec * 2)          # elementwise math
print("matrix shape:", mat.shape)
print("vector mean :", vec.mean())

# Pandas: labeled tabular data (rows = samples, columns = features)
df = pd.DataFrame({
    "feature": [5.1, 4.9, 6.3],
    "label":   [0, 0, 1],
})
print(df)
print("feature mean:", df["feature"].mean())

vector * 2 : [2 4 6]
matrix shape: (2, 2)
vector mean : 2.0
   feature  label
0      5.1      0
1      4.9      0
2      6.3      1
feature mean: 5.433333333333334


**2. What ML Actually Is**

Machine learning is about **learning a function from data** instead of hand-coding rules.

- **Features (X)** — the inputs describing each sample; **label (y)** — the answer you want to predict.
- **Supervised learning** — you have labels, and the model learns the mapping X → y (classification, regression).
- **Unsupervised learning** — no labels; the model finds structure on its own (clustering, dimensionality reduction).
- **Train/test split** — hold out data the model never sees during training, so you can measure how it performs on genuinely new inputs.

**3. The Core ML Workflow**

Almost every project follows the same loop:

`data → EDA (explore) → features → train → evaluate`

Load the data, explore it to understand distributions and issues, turn it into numeric features, fit a model on the training set, then evaluate on held-out data. The example below runs this end-to-end on the built-in Iris dataset.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# data -> features (X) and labels (y)
X, y = load_iris(return_X_y=True)

# train/test split - hold out data the model never sees during training
# Passing an integer (like 0 or 42) ensures your code produces the exact same random results—such as data splits or model weights—every single time you run it. 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# train
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# predict on unseen data
preds = model.predict(X_test)
print("predicted:", preds[:5])
print("actual   :", y_test[:5])

predicted: [1 0 2 1 1]
actual   : [1 0 2 1 1]


**4. Key Algorithms (Intuition)**

Focus on *when to use each*, not the math derivations:

| Algorithm | Intuition | Good for |
| --- | --- | --- |
| **Linear regression** | Fit a straight line/plane through the data | Predicting a continuous number |
| **Logistic regression** | A linear model squashed into a probability | Simple, interpretable classification |
| **Decision tree** | A flowchart of yes/no splits | Interpretable, non-linear patterns |
| **Random forest** | Many trees voting together | Strong general-purpose baseline |
| **k-NN** | Label a point by its nearest neighbours | Small datasets, simple boundaries |

**5. Evaluation**

"How do you know the model is good?" — the QA mindset's home turf. Accuracy alone can lie on imbalanced data, so look at several signals:

- **Accuracy** — fraction correct overall.
- **Precision** — of the items predicted positive, how many really were (cost of false positives).
- **Recall** — of the actual positives, how many you caught (cost of false negatives).
- **F1** — harmonic mean of precision and recall.
- **Confusion matrix** — the full breakdown of predicted vs actual.

In [3]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

# average='macro' averages the metric across all classes (multi-class case)
print(f"accuracy : {accuracy_score(y_test, preds):.3f}")
print(f"precision: {precision_score(y_test, preds, average='macro'):.3f}")
print(f"recall   : {recall_score(y_test, preds, average='macro'):.3f}")
print(f"f1       : {f1_score(y_test, preds, average='macro'):.3f}")
print("confusion matrix:\n", confusion_matrix(y_test, preds))

accuracy : 1.000
precision: 1.000
recall   : 1.000
f1       : 1.000
confusion matrix:
 [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


**6. Overfitting & Generalization**

The single most important idea in ML: a model that's perfect on training data can be useless on new data because it memorized noise instead of learning the pattern.

- **Overfitting** — low training error but high test error (too complex; high *variance*).
- **Underfitting** — high error everywhere (too simple; high *bias*).
- **Bias–variance tradeoff** — the balance between those two failure modes.
- **Cross-validation** — rotate which slice is held out across several folds for a more reliable score than a single split.

In [4]:
from sklearn.model_selection import cross_val_score

# cross-validation: rotate the held-out slice over several folds
# for a more reliable estimate than one train/test split
scores = cross_val_score(model, X, y, cv=5)
print("fold scores:", scores.round(3))
print(f"mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

fold scores: [0.967 1.    0.933 0.967 1.   ]
mean accuracy: 0.973 (+/- 0.025)
